# Colab PolicyServer + Tunnel Launch Instructions

Phase 11, Plan 11-03, Task 3. Documents how to stand up the Colab-hosted
`lerobot.async_inference.PolicyServer` and connect this project's local
`RobotClient` to it over a tunnel, before Plan 11-04 writes any code that
assumes this bridge exists.

**Selected checkpoint (Task 2's `checkpoint:decision`):** `victorvanhalst/smolvla_so101_cube`
— chosen for its exact task-instruction match to D-02's red-cube task. See
`checkpoint_candidates.md` for the fetched `config.json` data and
`11-03-SUMMARY.md` for the full decision rationale.

**pyngrok legitimacy gate:** this file's step 2 installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

---

## Camera mapping (Task 2's split-stereo resolution)

`victorvanhalst/smolvla_so101_cube`'s `config.json` expects 3 named camera
inputs (`camera1`, `camera2`, `camera3` — none marked `empty_camera_*`).
This project's real rig has only 2 physical cameras:

- IMX335 wrist camera, cv2 index 0, native 1920x1080
- AR0144 stereo camera, cv2 index 1, captured at 1600x600 (800x600 per
  half) — **one physical side-by-side stereo frame, not two independent
  feeds** (confirmed in `diagnostics/UAT/function/basic/UAT.md`, Step 4).
  1600x600 was chosen over the previously-implemented 1280x360 for
  meaningfully higher per-eye resolution while remaining confirmed stable;
  the still-broken 2560x720 mode remains frozen even after exhausting the
  last known macOS system-level fix — see `stereo_camera.py`'s module
  docstring for the full diagnosis.

Resolution: split the AR0144's single 1600x600 frame into left/right
halves, giving **3 genuinely distinct real camera views** (not a dummy or
duplicated 3rd slot):

| Checkpoint camera key | Real source                 | Crop                                         |
| --------------------- | --------------------------- | -------------------------------------------- |
| `camera1`           | IMX335 wrist                | as-is, full 1920x1080 frame                  |
| `camera2`           | AR0144 stereo — LEFT half  | columns`[0:800]` of the 1600x600 frame    |
| `camera3`           | AR0144 stereo — RIGHT half | columns`[800:1600]` of the 1600x600 frame |

This is a better structural fit than an empty/dummy 3rd slot: all 3 of the
checkpoint's expected inputs receive real image data during inference,
none is a zero/black placeholder. (Community precedent: other SmolVLA
fine-tunes such as `bklassen3434/smolvla_pick_pen_v2_frozen` and
`Yilin1001/smolvla-pen-v2-030000` pair a 2-real-camera rig with an
inherited 3rd config slot by leaving it a dummy — this project instead has
a 3rd genuinely distinct real view available via the stereo split, which
is a strictly better fit than a dummy slot would be.)

`lerobot.async_inference.robot_client`'s `--robot.cameras` flag takes a
dict of named camera configs, each independently opened by `cv2` — it has
no built-in "crop a region of a wider frame" option. Two ways to produce
`camera2`/`camera3` as independent feeds from the one AR0144 device:

1. **Preferred for this bridge:** a thin wrapper camera process/script that
   opens cv2 index 1 once, splits each frame into left/right halves, and
   republishes them as two separate virtual camera endpoints the
   `RobotClient`'s camera config can point at (e.g. two loopback
   `cv2.VideoCapture`-compatible sources, or if `lerobot`'s camera config
   supports a custom `type` plugin, register a "split-half" camera type
   that does the crop inline). This avoids opening the same physical
   device twice (most webcam drivers do not support concurrent opens of
   one index).
2. **Fallback if (1) isn't scriptable in time:** open cv2 index 1 once in
   the `RobotClient`'s local process, capture the full 1600x600 frame per
   step, crop it into `camera2`/`camera3` numpy arrays in the observation
   dict before it's sent to the `PolicyServer` — this requires a small
   patch/subclass of the stock `robot_client.py` observation-building step
   rather than pure CLI flags, since the stock CLI only supports
   independent `cv2` device opens per named camera.

Plan 11-04 (which writes `robot_client.py`-adjacent bridge code) is
responsible for implementing whichever of these two options it lands on;
this file only fixes the *mapping* (which checkpoint key gets which real
data), not the implementation mechanism.

---

In [1]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

OpenMP duplicate-runtime guards set.


## Step 1 — Colab: install `lerobot[async]`

In [2]:
!pip install 'lerobot[async]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 54.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 192.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 45.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.

Adds `grpcio` — already audited and approved in `11-RESEARCH.md`'s Package
Legitimacy Audit as a required transitive dependency of `lerobot`'s own
`async` extra. No separate checkpoint needed for this install.

## Step 2 — Colab: install `pyngrok` (gated — see legitimacy note above)

In [3]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

## Step 3 — Colab: start the `PolicyServer`

In [4]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 26.4 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=4b232ba9adabdfc30cf6d3ed40252a8abfaf24f56c595deb385fd58a6096022a
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [5]:
!pip install 'lerobot[dataset]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 83.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [6]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

Drive mount/token read skipped (MessageError: User cancelled dfs_ephemeral authorization)
Logged into HF Hub.
ngrok authtoken set.                                                                                


`lerobot`'s installs (Step 1's `lerobot[async]` and the earlier `lerobot[dataset]` cell) pull in a `torchaudio` build whose CUDA version doesn't always match Colab's preinstalled `torch` -- confirmed live (Phase 12 Test 7 UAT) to crash Step 3's `from lerobot.async_inference.policy_server import serve` with `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions`, which cascades into a `ModuleNotFoundError` for `transformers.AutoProcessor` and can leave the kernel needing a full restart. `torchaudio` isn't actually used by anything in this notebook -- uninstalling it removes the mismatched import path entirely (`transformers`' own `is_torchaudio_available()` check then correctly skips it).

In [7]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [8]:
import os
import socket
import subprocess
import sys
import time

POLICY_SERVER_LOG = "/content/policy_server.log"

# Test 7 UAT (Phase 12, 2026-09-27) root-cause work: `serve()` running as a
# Python threading.Thread inside THIS kernel process meant a process-level
# crash (OOM-kill, CUDA driver segfault) took the whole kernel down with it
# -- Flask depth server, both tunnels, everything -- leaving only "The Kernel
# crashed while executing code in the current cell or a previous cell", not a
# catchable Python exception. The saved cell output also showed the crash
# happened mid-load of SmolVLA's VLM backbone, in the SAME kernel process
# where Step 7 later does `pip install torch==2.6.0 ... --index-url
# .../cu124` -- a version/CUDA-build reinstall happening AFTER this cell had
# already imported the original torch into memory.
#
# Launching PolicyServer as a genuine, separate OS subprocess instead means:
#   - a crash in it cannot take this kernel (or the depth server/tunnels) down
#   - its stdout/stderr go straight to a file via the OS, independent of
#     Jupyter's per-cell output capture, so the log survives even a hard crash
#   - it re-imports torch fresh from disk at subprocess start, so it can
#     never end up running on a stale in-process torch that predates a later
#     `pip install torch==...` -- each subprocess only ever sees whatever is
#     on disk when IT starts, fully isolated from Step 7's reinstall
#   - CUDA_LAUNCH_BLOCKING=1 forces synchronous CUDA execution, so an async
#     driver-level crash surfaces as a normal Python traceback in the log
#     instead of an opaque process death
#   - KMP_DUPLICATE_LIB_OK / OMP_NUM_THREADS (also set globally above) guard
#     this subprocess too against the "OMP: Error #15" duplicate-runtime crash

# Kill any stray PolicyServer/run_demo.py processes left over from a
# previous attempt tonight -- a hung child from an earlier crash keeps
# holding GPU memory, which can itself cause the NEXT model load to OOM.
subprocess.run(["pkill", "-9", "-f", "lerobot.async_inference.policy_server"], check=False)
subprocess.run(["pkill", "-9", "-f", "scripts/run_demo.py"], check=False)
time.sleep(1)

print("--- GPU memory before starting PolicyServer ---")
subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"],
    check=False,
)

_env = os.environ.copy()
_env["CUDA_LAUNCH_BLOCKING"] = "1"
_env.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
_env.setdefault("OMP_NUM_THREADS", "1")

_log_fh = open(POLICY_SERVER_LOG, "w", buffering=1)
policy_server_process = subprocess.Popen(
    [
        sys.executable, "-m", "lerobot.async_inference.policy_server",
        "--host", "0.0.0.0", "--port", "5173",
    ],
    stdout=_log_fh,
    stderr=subprocess.STDOUT,
    env=_env,
)

# Wait for the server to actually be listening before Step 4 opens the
# tunnel against it (subprocess startup is not instant).
for _ in range(60):
    if policy_server_process.poll() is not None:
        raise RuntimeError(
            f"PolicyServer subprocess exited early (code {policy_server_process.returncode}) "
            f"-- see {POLICY_SERVER_LOG}"
        )
    try:
        with socket.create_connection(("127.0.0.1", 5173), timeout=1):
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(f"PolicyServer did not start listening in time -- see {POLICY_SERVER_LOG}")

print(f"PolicyServer running as PID {policy_server_process.pid}. Logs -> {POLICY_SERVER_LOG}")
print(f"Tail it live from another cell with: !tail -f {POLICY_SERVER_LOG}")
print("Check it's still alive from another cell with: policy_server_process.poll()  # None = still running")

--- GPU memory before starting PolicyServer ---
PolicyServer running as PID 2006. Logs -> /content/policy_server.log
Tail it live from another cell with: !tail -f /content/policy_server.log
Check it's still alive from another cell with: policy_server_process.poll()  # None = still running


PolicyServer now runs as a real OS subprocess (`policy_server_process`), not
a background thread -- Colab's async-cell execution note above no longer
applies, but the same idea holds: this cell returns immediately once the
server is confirmed listening, so the notebook can go straight on to the
tunnel cell below while the subprocess keeps running independently.

## Step 4 — Colab: open a TCP tunnel (not HTTP)

In [9]:
# Colab notebook cell
from pyngrok import ngrok

# Explicitly request a TCP tunnel — gRPC needs HTTP/2 semantics that a
# default HTTP tunnel silently breaks (Pitfall 5 in 11-RESEARCH.md).
tunnel = ngrok.connect(5173, "tcp")
print(f"PolicyServer reachable at: {tunnel.public_url}")
# public_url looks like tcp://0.tcp.ngrok.io:12345 — strip the "tcp://"
# prefix; the local RobotClient's --server_address flag wants host:port.

PolicyServer reachable at: tcp://0.tcp.ngrok.io:13306


In [10]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True NVIDIA L4


In [11]:
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total --format=csv

utilization.gpu [%], utilization.memory [%], memory.used [MiB], memory.total [MiB]
0 %, 0 %, 3 MiB, 23034 MiB


Copy the printed `host:port` (after stripping `tcp://`) for Step 5.

If ngrok's free tier no longer supports a TCP tunnel endpoint by the time
this is run (Assumption A4 in `11-RESEARCH.md`), fall back to `cloudflared`
with a configured Zero Trust tunnel for raw TCP forwarding (heavier setup
than ngrok TCP — cloudflared's zero-config "quick tunnels" are HTTP-only
and will not work here).

## Step 5 — Local (`control/`): launch the `RobotClient`

Notes:

- `<ngrok_tcp_host>:<ngrok_tcp_port>` = the value printed in Step 4, with
  the `tcp://` prefix stripped.
- `--robot.port` / `--robot.id` match `control/COMMANDS.md`'s documented
  follower arm (`/dev/cu.usbmodem5B8E1139151`, `soarm_follower_02`) — reverify
  via `ls /dev/cu.usbmodem*` if the port has been reassigned since last use.
- The `camera2`/`camera3` `type: <split-stereo-left/right>` placeholders are
  **not** stock `lerobot` camera types — see the "Camera mapping" section
  above; Plan 11-04 implements the actual split mechanism (custom camera
  type plugin, or a `robot_client.py` observation-patch). This file
  documents the target mapping, not that implementation.
- `--task` uses the exact instruction wording from the checkpoint's own
  training task ("Pick the red cube and place it in the bowl"), not a
  paraphrase — the model card / task description string should be
  reverified against the actual checkpoint metadata before the live run in
  Plan 11-05, since exact wording can matter for language-conditioned
  policies.
- `--policy_device=cuda` assumes the `PolicyServer` runs on a Colab GPU
  runtime; the policy device is a Colab-side concern (loaded by
  `PolicyServerConfig`/`serve()` in Step 3), this flag on the local CLI
  side only affects what the `RobotClient` requests be reported.

## Step 6 — Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted `PolicyServer` is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day:

In [26]:
# Colab notebook cell, at the end of the session
ngrok.disconnect(tunnel.public_url)
ngrok.kill()

# PolicyServer now runs as a real subprocess (Step 3 fix, 2026-09-27), not a
# background thread -- terminate it explicitly so it doesn't keep holding
# GPU memory after the tunnel is torn down.
if "policy_server_process" in dir() and policy_server_process.poll() is None:
    policy_server_process.terminate()
    try:
        policy_server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        policy_server_process.kill()
    print("PolicyServer subprocess terminated.")

The cell above now terminates the `PolicyServer` subprocess directly
(`policy_server_process.terminate()`); no need to click a cell's stop button
or restart the runtime for it, since it's no longer running inside the
notebook kernel itself (2026-09-27 subprocess-isolation fix).

---

### Step 7 — Colab: install and load Fast-FoundationStereo

In [12]:
# Colab notebook cell -- DO NOT RUN until the human has completed the
# Fast-FoundationStereo legitimacy check above.
!git clone https://github.com/NVlabs/Fast-FoundationStereo

Cloning into 'Fast-FoundationStereo'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 274 (delta 55), reused 31 (delta 30), pack-reused 188 (from 1)
Receiving objects: 100% (274/274), 154.87 MiB | 19.82 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [14]:
# Colab notebook cell -- gated by the same Fast-FoundationStereo legitimacy
# check above (do not run before that check and before Step 7's clone cell).
# FastFS's documented "Option 2: pip" environment setup (docs/Fast Foundation
# Stereo Readme.md, "Environment setup" section), run inside the just-cloned
# repo directory.
#
# Root-caused 2026-09-29 (Test 7 UAT retry, PolicyServer subprocess exit code
# -6 / SIGABRT, confirmed via /content/policy_server.log timeline): cuDNN 9.x
# splits into several plugin .so files (libcudnn_graph.so, libcudnn_ops.so,
# ...) that PyTorch dlopen's LAZILY on first real use, not at import time.
# PolicyServer's subprocess (Step 3) imports torch and loads the model fine,
# but doesn't touch the cudnn_graph plugin until the very first real
# inference call. Because THIS cell overwrites the shared filesystem's
# on-disk cuDNN files with a different pinned version (torch==2.6.0+cu124 ->
# nvidia-cudnn-cu12==9.1.0.70) while that subprocess is still running, its
# lazy dlopen at first-inference-time picks up the NEW mismatched files,
# producing an unresolved-symbol native abort ("undefined symbol:
# cudnnGetLibConfig") deep inside the first inference call. Process-level
# isolation (Step 3's subprocess fix) does not protect against this on its
# own, since both processes still share one filesystem. Installing FastFS's
# torch build into its own venv, fully separate from the ambient
# site-packages PolicyServer's subprocess reads from, removes the
# shared-file race entirely -- this cell can now run before, after, or be
# re-run independently of Step 3/6 with no interaction at all.
#
# Uses `virtualenv` (pip-installed, bundles its own pip/setuptools/wheel),
# NOT the stdlib `venv` module -- Colab's OS-level `apt` `python3` alias is
# 3.10 while the actual running Python is 3.12, so `apt-get install
# python3-venv` grabs the WRONG (and apparently now 404'ing) package and
# leaves `venv`'s bundled ensurepip broken. `virtualenv` needs no OS package
# at all, so this class of failure can't recur.
%cd /content/Fast-FoundationStereo

FASTFS_VENV = "/content/fastfs_venv"
FASTFS_PYTHON = f"{FASTFS_VENV}/bin/python"

!pip install -q virtualenv
!python -m virtualenv {FASTFS_VENV}
!{FASTFS_PYTHON} -m pip install --upgrade pip
!{FASTFS_PYTHON} -m pip install torch==2.6.0 torchvision==0.21.0 xformers --index-url https://download.pytorch.org/whl/cu124
!{FASTFS_PYTHON} -m pip install -r requirements.txt

# `scripts/run_demo.py` calls two blocking, display-dependent GUI paths that
# are NOT gated by any CLI flag and hang/crash forever on a headless Colab
# runtime (confirmed live during Phase 12 Test 6 UAT):
#   1. `cv2.imshow('disp', ...)` + `cv2.waitKey(0)` -- unconditional, runs on
#      every invocation regardless of flags. The disparity visualization is
#      already saved to `{out_dir}/disp_vis.png` via `imageio.imwrite` right
#      before this call, so commenting it out loses no output.
#   2. The Open3D interactive viewer (`o3d.visualization.Visualizer()` /
#      `vis.run()` / `vis.destroy_window()`), which only runs when
#      `--get_pc 1` -- required below to get `depth_meter.npy` written at
#      all (see Step 8's `run_fastfs_inference()`). The point cloud is
#      already saved to `{out_dir}/cloud.ply` via `o3d.io.write_point_cloud`
#      right before this block, so commenting it out loses no output either.
# Pattern-based (not line-number-based) so this survives upstream edits to
# `run_demo.py` as long as these call sites keep their current text.
!sed -i "/cv2.imshow('disp'/,/cv2.waitKey(0)/ s/^/#/" scripts/run_demo.py
!sed -i '/Visualizing point cloud/,/vis.destroy_window()/ s/^/#/' scripts/run_demo.py

/content/Fast-FoundationStereo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 145.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 62.2 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 379ms
  creator CPython3Posix(dest=/content/fastfs_venv, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 129.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 284.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.2 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66

Then manually download the `23-36-37` checkpoint from the README's Google
Drive folder (https://drive.google.com/drive/folders/1HuTt7UIp7gQsMiDvJwVuWmKpvFzIIMap)
and place `model_best_bp2_serialize.pth` at
`/content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth`
in the Colab runtime -- a manual human step (a gated/shared Google Drive
folder isn't reliably scriptable inside a single Colab cell), not something
this notebook automates.

Step 8 below invokes FastFS's own documented `scripts/run_demo.py` CLI
(README's "Run demo" section) directly, so the input/output CONTRACT (flags,
intrinsic-file format) is fully documented there -- no fabricated
function/class names. `run_demo.py`'s exact output filename inside
`--out_dir` is confirmed (live, during Phase 12 Test 6 UAT) to be
`depth_meter.npy`, written only when `--get_pc 1` -- Step 8's
`DEPTH_OUTPUT_GLOB` and `--get_pc` flag reflect this.

Colab: upload the downloaded checkpoint (run this cell after downloading `model_best_bp2_serialize.pth` to your own computer from the Drive link above):

In [15]:
import os

os.makedirs("/content/Fast-FoundationStereo/weights/23-36-37", exist_ok=True)

!pip install -q gdown
!gdown "1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth
!gdown "1GDBRYL-ZaLpXEtWfGFRJvkBc_2sywjgj" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/cfg.yaml

print(os.listdir("/content/Fast-FoundationStereo/weights/23-36-37"))

Downloading...
From (original): https://drive.google.com/uc?id=1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E
From (redirected): https://drive.google.com/uc?id=1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E&confirm=t&uuid=7c6866a2-fc25-4db1-b7c4-0eecdacd2b6c
To: /content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth
100% 71.1M/71.1M [00:00<00:00, 87.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GDBRYL-ZaLpXEtWfGFRJvkBc_2sywjgj
To: /content/Fast-FoundationStereo/weights/23-36-37/cfg.yaml
100% 182/182 [00:00<00:00, 965kB/s]
['model_best_bp2_serialize.pth', 'cfg.yaml']


### Step 8 — Colab: serve the depth endpoint

In [ ]:
!pip install flask

import base64
import glob
import io
import os
import subprocess
import tempfile
from pathlib import Path

import cv2
import numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

FASTFS_DIR = "/content/Fast-FoundationStereo"
FASTFS_MODEL = f"{FASTFS_DIR}/weights/23-36-37/model_best_bp2_serialize.pth"
# Step 7 now installs FastFS's own torch/cuDNN build into an isolated venv
# (2026-09-29 fix) instead of the ambient site-packages PolicyServer's
# subprocess reads from -- invoke run_demo.py with THAT interpreter so this
# subprocess's torch/cuDNN never touches or gets touched by PolicyServer's.
FASTFS_PYTHON = "/content/fastfs_venv/bin/python"

# Confirmed live (Phase 12 Test 6 UAT): run_demo.py only writes this file
# when `--get_pc 1` is passed (see the subprocess args below) -- with
# `--get_pc 0`, no .npy is ever written and this glob matches nothing.
DEPTH_OUTPUT_GLOB = "depth_meter.npy"

# Test 7 UAT crash investigation (2026-09-27): this subprocess uses whatever
# torch Step 7 just installed (torch==2.6.0+cu124), on the SAME physical GPU
# that PolicyServer's subprocess (Step 3) separately loads SmolVLA onto. If a
# prior FastFS call hangs or is killed uncleanly, its CUDA context can be
# left holding VRAM, starving the next model load -- log GPU memory around
# every call so a leak here is visible immediately, and bound the subprocess
# with a timeout so a hang can't pin GPU memory indefinitely. Also pass the
# same OpenMP duplicate-runtime guards set globally above, since this
# subprocess gets a fresh environment copy, not inherited process state.
_fastfs_env = os.environ.copy()
_fastfs_env.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
_fastfs_env.setdefault("OMP_NUM_THREADS", "1")


def _log_gpu_memory(label):
    print(f"--- GPU memory {label} ---")
    subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"],
        check=False,
    )


def run_fastfs_inference(left_img, right_img, intrinsics_flat, baseline_m):
    """Invokes Fast-FoundationStereo's documented `scripts/run_demo.py` CLI
    (docs/Fast Foundation Stereo Readme.md, "Run demo" section) as a
    subprocess, then loads the resulting depth map back into this process."""
    with tempfile.TemporaryDirectory() as tmp_dir:
        left_path = str(Path(tmp_dir) / "left.png")
        right_path = str(Path(tmp_dir) / "right.png")
        intrinsic_path = str(Path(tmp_dir) / "K.txt")
        out_dir = str(Path(tmp_dir) / "out")
        Path(out_dir).mkdir(parents=True, exist_ok=True)

        cv2.imwrite(left_path, left_img)
        cv2.imwrite(right_path, right_img)

        # README: line 1 = flattened 1x9 intrinsics matrix (space-separated),
        # line 2 = baseline in meters.
        with open(intrinsic_path, "w") as f:
            f.write(" ".join(str(v) for v in intrinsics_flat) + "\n")
            f.write(f"{baseline_m}\n")

        _log_gpu_memory("before FastFS inference")
        try:
            subprocess.run(
                [
                    FASTFS_PYTHON, "scripts/run_demo.py",
                    "--model_dir", FASTFS_MODEL,
                    "--left_file", left_path,
                    "--right_file", right_path,
                    "--intrinsic_file", intrinsic_path,
                    "--out_dir", out_dir,
                    "--remove_invisible", "0",
                    "--denoise_cloud", "0",
                    "--scale", "1",
                    "--get_pc", "1",
                    "--valid_iters", "8",
                    "--max_disp", "192",
                    "--zfar", "100",
                ],
                cwd=FASTFS_DIR,
                env=_fastfs_env,
                check=True,
                timeout=120,
            )
        finally:
            _log_gpu_memory("after FastFS inference")

        npy_files = sorted(glob.glob(str(Path(out_dir) / DEPTH_OUTPUT_GLOB)))
        if not npy_files:
            raise RuntimeError(
                f"No files matching {DEPTH_OUTPUT_GLOB!r} found in {out_dir} "
                "after run_demo.py -- check that scripts/run_demo.py in this "
                "runtime still has Step 7's headless-GUI patch applied."
            )
        return np.load(npy_files[0])


@app.route("/depth", methods=["POST"])
def depth():
    body = request.get_json()
    left_bytes = base64.b64decode(body["left_png_b64"])
    right_bytes = base64.b64decode(body["right_png_b64"])
    left_img = cv2.imdecode(np.frombuffer(left_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    right_img = cv2.imdecode(np.frombuffer(right_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    intrinsics_flat = body["intrinsics_flat"]
    baseline_m = body["baseline_m"]

    depth_map = run_fastfs_inference(left_img, right_img, intrinsics_flat, baseline_m)

    buf = io.BytesIO()
    np.save(buf, depth_map.astype(np.float32))
    depth_npy_b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return jsonify({"depth_npy_b64": depth_npy_b64})


# Run in the background, same convention as Step 3's PolicyServerConfig/serve() cell.
import threading
threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=3000, use_reloader=False),
    daemon=True,
).start()
print("Flask depth server started in background on port 3000")

Flask depth server started in background on port 3000


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:3000
 * Running on http://172.30.0.2:3000
INFO:werkzeug:Press CTRL+C to quit


This exact request/response shape (`left_png_b64`/`right_png_b64`/
`intrinsics_flat`/`baseline_m` request, `depth_npy_b64` response) is what
`depth_camera.DepthCameraClient.compute_depth()` (Plan 12-06 Task 1) sends
and parses -- keep them in sync if either side changes.

### Step 9 — Colab: open a second tunnel

In [17]:
from pyngrok import ngrok

# A DIFFERENT local port than Step 3's gRPC server (5173), and a plain HTTP
# tunnel type -- unlike Step 4's gRPC tunnel, which specifically needs "tcp"
# for HTTP/2 semantics, this is a plain JSON-over-HTTP POST endpoint, so
# ngrok's default HTTP tunnel type works fine here.
depth_tunnel = ngrok.connect(3000, "http")
print(f"Depth endpoint reachable at: {depth_tunnel.public_url}/depth")

Depth endpoint reachable at: https://runway-uninsured-blank.ngrok-free.dev/depth


--- GPU memory before FastFS inference ---


INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 11:04:03] "POST /depth HTTP/1.1" 200 -


--- GPU memory after FastFS inference ---
--- GPU memory before FastFS inference ---


INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 11:04:18] "POST /depth HTTP/1.1" 200 -


...

--- GPU memory after FastFS inference ---
--- GPU memory before FastFS inference ---


INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 11:06:22] "POST /depth HTTP/1.1" 200 -


--- GPU memory after FastFS inference ---
--- GPU memory before FastFS inference ---


INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 11:06:41] "POST /depth HTTP/1.1" 200 -


--- GPU memory after FastFS inference ---


Pass the printed `.../depth` URL as `depth_camera.py`'s `--endpoint` argument
(Plan 12-06 Task 1) or `run_vla_episode.py`'s `--depth-endpoint` flag
(Task 3).

### Teardown (extends Step 6 above)

Every session, no exceptions, ALSO tear down this second tunnel and stop the
Flask server process -- same rationale as the existing gRPC tunnel's own
teardown discipline (an unattended open tunnel into a Colab-hosted endpoint
is an elevation-of-privilege risk):

In [25]:
ngrok.disconnect(depth_tunnel.public_url)
ngrok.kill() #already stops all tunnels including this one, if not already called above.

Then interrupt/stop the Flask `app.run(...)` cell (Colab: click the cell's
stop button, or restart the runtime) so the depth server process itself is
not left listening after the tunnel is torn down.

---

## Summary

| Item             | Value                                                                                                                            |
| ---------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| Checkpoint       | `victorvanhalst/smolvla_so101_cube`                                                                                            |
| Task instruction | "Pick the red cube and place it in the bowl"                                                                                     |
| Tunnel type      | TCP (`ngrok.connect(5173, "tcp")`) — not HTTP                                                                                 |
| Camera mapping   | `camera1`=wrist, `camera2`=AR0144 stereo-left, `camera3`=AR0144 stereo-right                                               |
| Fallback tunnel  | `cloudflared` + configured Zero Trust tunnel (heavier setup)                                                                   |
| Fallback policy  | ACT or pi0/pi05 (`lerobot.policies`, already installed) if this checkpoint underperforms                                       |
| Depth endpoint   | Fast-FoundationStereo (NVlabs/Fast-FoundationStereo), served via Flask, HTTP tunnel (separate from the gRPC PolicyServer tunnel) |

In [18]:
!tail -100 /content/policy_server.log

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-09-29 10:51:52 y_server.py:420 {'fps': 30,
 'host': '0.0.0.0',
 'inference_latency': 0.03333333333333333,
 'obs_queue_timeout': 2,
 'port': 5173}
INFO 2026-09-29 10:51:52 y_server.py:430 PolicyServer started on 0.0.0.0:5173
INFO 2026-09-29 11:02:56 y_server.py:110 Client ipv6:%5B::1%5D:35870 connected and ready
INFO 2026-09-29 11:02:56 y_server.py:136 Receiving policy instructions from ipv6:%5B::1%5D:35870 | Policy type: smolvla | Pretrained name or path: victorvanhalst/smolvla_so101_cube | Actions per chunk: 50 | Device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary

In [20]:
!pip show torch nvidia-cudnn-cu12 | grep -E "^(Name|Version)"
!python -c "import torch; print('torch', torch.__version__); print('cudnn', torch.backends.cudnn.version())"

Name: torch
Version: 2.11.0+cu128
Name: nvidia-cudnn-cu12
Version: 9.19.0.56
torch 2.11.0+cu128
cudnn 91900


In [19]:
!{FASTFS_PYTHON} -c "import torch; print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

cuda available: True
device: NVIDIA L4


In [21]:
policy_server_process.poll()

In [22]:
!tail -40 /content/policy_server.log

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
INFO 2026-09-29 10:51:52 y_server.py:420 {'fps': 30,
 'host': '0.0.0.0',
 'inference_latency': 0.03333333333333333,
 'obs_queue_timeout': 2,
 'port': 5173}
INFO 2026-09-29 10:51:52 y_server.py:430 PolicyServer started on 0.0.0.0:5173
INFO 2026-09-29 11:02:56 y_server.py:110 Client ipv6:%5B::1%5D:35870 connected and ready
INFO 2026-09-29 11:02:56 y_server.py:136 Receiving policy instructions from ipv6:%5B::1%5D:35870 | Policy type: smolvla | Pretrained name or path: victorvanhalst/smolvla_so101_cube | Actions per chunk: 50 | Device: cuda
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary